In [4]:
!pip install transformers sentencepiece accelerate -q

In [5]:
import pandas as pd
from tqdm import tqdm

from transformers import pipeline

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)


In [6]:
train_df = pd.read_csv(
    "train.csv",
    engine="python",
    on_bad_lines="skip"
)

prompt_df = pd.read_csv(
    "prompting_test.csv",
    engine="python",
    on_bad_lines="skip"
)


# prompt_df = prompt_df.head(100)

In [7]:
train_df["label"] = train_df["label"].map({
    "FAKE": 0,
    "REAL": 1
})

prompt_df["label"] = prompt_df["label"].map({
    "FAKE": 0,
    "REAL": 1
})

In [8]:

prompt_df["text"] = (
    prompt_df["text"]
    .astype(str)
    .str[:300]
)

In [9]:
# LOAD MODEL

classifier = pipeline(
    "text-generation",
    model="google/flan-t5-small",
    device=-1,
    max_new_tokens=2
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


...

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DiffLlamaF

In [11]:
# FEW-SHOT EXAMPLES
fake_example = train_df[
    train_df["label"] == 0
].iloc[0]["text"][:100]

real_example = train_df[
    train_df["label"] == 1
].iloc[0]["text"][:100]

# FEW-SHOT PROMPT

def few_shot_prompt(text):

    return f"""
Fake: {fake_example}

Real: {real_example}

{text}

REAL or FAKE?
"""


# PREDICT FUNCTION
def predict_few_shot(text):

    prompt = few_shot_prompt(text)

    result = classifier(prompt)[0]["generated_text"]

    result = result.strip().upper()

    if "REAL" in result:
        return 1
    else:
        return 0


# PREDICTION
few_predictions = []

for text in tqdm(prompt_df["text"]):

    pred = predict_few_shot(text)

    few_predictions.append(pred)

# EVALUATION

y_true = prompt_df["label"]
y_pred = few_predictions

print("Accuracy :", round(accuracy_score(y_true, y_pred), 4))
print("Precision:", round(precision_score(y_true, y_pred), 4))
print("Recall   :", round(recall_score(y_true, y_pred), 4))
print("F1-score :", round(f1_score(y_true, y_pred), 4))
print("ROC-AUC  :", round(roc_auc_score(y_true, y_pred), 4))

# SAVE RESULT


prompt_df["few_shot_pred"] = few_predictions

prompt_df.to_csv("few_shot_results.csv", index=False)

print("Saved: few_shot_results.csv")

100%|██████████| 500/500 [04:43<00:00,  1.76it/s]


Accuracy : 0.648
Precision: 0.648
Recall   : 1.0
F1-score : 0.7864
ROC-AUC  : 0.5
Saved: few_shot_results.csv
